In [2]:
import subprocess
import sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'qiskit==2.2.3','qiskit-machine-learning==0.9.0','numpy==2.2.6',
    'scipy==1.15.3','scikit-learn==1.6.1','pandas==2.2.3',
    'matplotlib==3.10.1','flask==3.1.0'])

0

In [8]:
import numpy as np
x=np.array([550000,8,.82,.76,.35,1,1])
signals=np.array([x[0]>220000,x[1]<30,x[2]>.5,x[3]>.5,x[4]<.6,x[5]==1,x[6]==1],int)
print('위험 신호:',signals.tolist())
print('신호 합/라벨:',int(signals.sum()),int(signals.sum()>=3))

위험 신호: [1, 1, 1, 1, 1, 1, 1]
신호 합/라벨: 7 1


In [9]:
import time
from pathlib import Path
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from qiskit.circuit.library import z_feature_map, zz_feature_map, pauli_feature_map
from qiskit.circuit.library import real_amplitudes, efficient_su2, n_local, pauli_two_design
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.primitives import QMLSampler, QMLEstimator
from qiskit_machine_learning.neural_networks import SamplerQNN, EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import VQC

def parity(integer):
    return int(integer).bit_count() % 2

def circuits(n, fmap_name='Z', ansatz_name='RealAmplitudes', reps=1):
    if fmap_name == 'Z':
        fmap = z_feature_map(n, reps=1)
    elif fmap_name == 'ZZ':
        fmap = zz_feature_map(n, reps=1, entanglement='linear')
    elif fmap_name == 'Pauli':
        fmap = pauli_feature_map(n, reps=1, paulis=['Z','XX'], entanglement='linear')
    else:
        raise ValueError(fmap_name)
    if ansatz_name == 'RealAmplitudes':
        ansatz = real_amplitudes(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'EfficientSU2':
        ansatz = efficient_su2(n, reps=reps, entanglement='linear')
    elif ansatz_name == 'TwoLocal':
        ansatz = n_local(n, rotation_blocks=['ry','rz'], entanglement_blocks='cz', reps=reps, entanglement='linear')
    elif ansatz_name == 'PauliTwoDesign':
        ansatz = pauli_two_design(n, reps=reps, seed=33)
    else:
        raise ValueError(ansatz_name)
    return fmap, ansatz

def fit_model(kind, train_x, train_y, fmap_name='Z', ansatz_name='RealAmplitudes', maxiter=40, reps=1):
    fmap, ansatz = circuits(train_x.shape[1], fmap_name, ansatz_name, reps)
    initial = np.linspace(0.1,0.7,ansatz.num_parameters)
    history = []
    def optimizer(fun, x0, jac=None, bounds=None):
        def tracked(weights):
            value = float(fun(weights))
            history.append(value)
            return value
        return minimize(tracked, x0, method='COBYLA', options={'maxiter':maxiter, 'rhobeg':0.3})
    started = time.perf_counter()
    if kind == 'VQC':
        model = VQC(feature_map=fmap, ansatz=ansatz, sampler=QMLSampler(shots=None),
                    interpret=parity, output_shape=2, initial_point=initial, optimizer=optimizer)
        model.fit(train_x, train_y)
        def predict_probability(x):
            return np.asarray(model.neural_network.forward(x, model.weights))[:,1]
        weights = model.weights
    else:
        circuit = fmap.compose(ansatz)
        if kind == 'SQNN':
            qnn = SamplerQNN(circuit=circuit, sampler=QMLSampler(shots=None),
                             input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                             interpret=parity, output_shape=2)
            def probability(x,w):
                return np.asarray(qnn.forward(x,w))[:,1]
            start = initial
        elif kind == 'EQNN':
            qnn = EstimatorQNN(circuit=circuit, estimator=QMLEstimator(default_precision=0),
                              observables=SparsePauliOp('Z'*train_x.shape[1]),
                              input_params=list(fmap.parameters), weight_params=list(ansatz.parameters),
                              default_precision=0)
            def probability(x,w):
                expectation = np.asarray(qnn.forward(x,w[:-2])).ravel()
                # 마지막 두 값은 고전 출력층의 가중치와 편향입니다.
                score = np.clip(w[-2]*expectation+w[-1],-40,40)
                return 1/(1+np.exp(-score))
            start = np.r_[initial,-1.0,0.0]
        else:
            raise ValueError(kind)
        def loss(w):
            p1 = np.clip(probability(train_x,w),1e-10,1-1e-10)
            value = float(-np.mean(train_y*np.log2(p1)+(1-train_y)*np.log2(1-p1)))
            return value
        result = optimizer(loss,start)
        weights = result.x
        def predict_probability(x):
            return probability(x,weights)
        model = None
    print(kind, fmap_name, ansatz_name, '학습 평가 수:',len(history),
          '최초/마지막 기록 손실:',round(history[0],6),round(history[-1],6))
    return predict_probability, model, {'parameters':len(weights),'depth':fmap.compose(ansatz).depth(),
                                       'seconds':time.perf_counter()-started,'history':history}

def report(name,y,p1):
    p1 = np.asarray(p1).ravel()
    assert p1.shape == y.shape and np.isfinite(p1).all()
    assert np.all((p1>=-1e-8)&(p1<=1+1e-8))
    predicted = (p1>=0.5).astype(int)
    result = {'model':name, 'accuracy':accuracy_score(y,predicted),
              'precision':precision_score(y,predicted,zero_division=0),
              'recall':recall_score(y,predicted,zero_division=0),
              'f1':f1_score(y,predicted,zero_division=0)}
    print({k:round(float(v),4) if k!='model' else v for k,v in result.items()})
    print('혼동행렬 [[TN,FP],[FN,TP]]:',confusion_matrix(y,predicted,labels=[0,1]).tolist())
    return result

def split_fixed(X,y):
    # 각 클래스에서 원래 행 순서의 네 번째마다 시험 표본으로 사용합니다.
    test = np.concatenate([np.flatnonzero(y==c)[3::4] for c in [0,1]])
    train = np.setdiff1d(np.arange(len(y)),test)
    assert not np.intersect1d(train,test).size
    assert len(np.unique(y[train]))==len(np.unique(y[test]))==2
    return train,test

print('버전:',{k:metadata.version(k) for k in ['qiskit','qiskit-machine-learning','numpy','scipy','scikit-learn']})

X = np.array([[8000.0, 240.0, 0.05, 0.1, 0.95, 0.0, 0.0],
 [900000.0, 5.0, 0.85, 0.8, 0.2, 1.0, 1.0],
 [50000.0, 180.0, 0.15, 0.2, 0.9, 0.0, 0.0],
 [550000.0, 8.0, 0.82, 0.76, 0.35, 1.0, 1.0],
 [120000.0, 600.0, 0.2, 0.15, 0.85, 0.0, 0.0],
 [300000.0, 15.0, 0.6, 0.7, 0.4, 1.0, 0.0],
 [180000.0, 90.0, 0.25, 0.2, 0.8, 0.0, 0.0],
 [750000.0, 20.0, 0.75, 0.85, 0.3, 0.0, 1.0],
 [250000.0, 300.0, 0.15, 0.2, 0.9, 0.0, 0.0],
 [450000.0, 10.0, 0.7, 0.6, 0.5, 1.0, 1.0],
 [60000.0, 25.0, 0.1, 0.2, 0.9, 0.0, 0.0],
 [100000.0, 5.0, 0.8, 0.7, 0.4, 1.0, 0.0],
 [130000.0, 120.0, 0.3, 0.35, 0.75, 0.0, 0.0],
 [600000.0, 40.0, 0.65, 0.75, 0.45, 1.0, 1.0],
 [40000.0, 500.0, 0.1, 0.1, 0.95, 1.0, 0.0],
 [1500000.0, 2.0, 0.95, 0.9, 0.2, 1.0, 1.0],
 [190000.0, 60.0, 0.25, 0.3, 0.85, 0.0, 1.0],
 [240000.0, 50.0, 0.55, 0.6, 0.55, 1.0, 0.0],
 [10000.0, 1440.0, 0.05, 0.05, 0.95, 0.0, 0.0],
 [350000.0, 18.0, 0.4, 0.7, 0.35, 0.0, 1.0],
 [220000.0, 35.0, 0.4, 0.4, 0.7, 0.0, 0.0],
 [500000.0, 7.0, 0.8, 0.45, 0.3, 1.0, 0.0],
 [160000.0, 200.0, 0.2, 0.25, 0.88, 0.0, 0.0],
 [280000.0, 12.0, 0.6, 0.65, 0.5, 0.0, 1.0]], dtype=float)
y = ((X[:,0]>220000).astype(int)+(X[:,1]<30).astype(int)
     +(X[:,2]>.5).astype(int)+(X[:,3]>.5).astype(int)+(X[:,4]<.6).astype(int)
     +X[:,5].astype(int)+X[:,6].astype(int)>=3).astype(int)
train,test = split_fixed(X,y)
scaler = MinMaxScaler(feature_range=(0,1),clip=True).fit(X[train])
train_x,test_x = scaler.transform(X[train]),scaler.transform(X[test])
print('학습/시험 크기:',train_x.shape,test_x.shape)
print('학습/시험 라벨 수:',np.bincount(y[train]).tolist(),np.bincount(y[test]).tolist())
results=[]
for fmap in ['Z','ZZ','Pauli']:
    for ansatz in ['RealAmplitudes','EfficientSU2','TwoLocal','PauliTwoDesign']:
        for kind in ['VQC','SQNN','EQNN']:
            predict,model,info=fit_model(kind,train_x,y[train],fmap,ansatz,maxiter=40)
            row=report(kind,y[test],predict(test_x))
            row.update(feature_map=fmap,ansatz=ansatz,parameters=info['parameters'],depth=info['depth'])
            results.append(row)
table=pd.DataFrame(results)
table.to_csv('qml_fixed_results.csv',index=False)
summary=table.groupby('model')['f1'].agg(['min','max','mean']).round(4)
print('모델별 F1 요약:');print(summary.to_string())
fig,ax=plt.subplots(figsize=(12,5))
for name,group in table.groupby('model',sort=False):
    ax.plot(np.arange(len(group)),group['f1'],marker='o',label=name)
labels=[f'{f}/{a}' for f in ['Z','ZZ','Pauli'] for a in ['RealAmplitudes','EfficientSU2','TwoLocal','PauliTwoDesign']]
ax.set_xticks(np.arange(12),labels,rotation=70,ha='right')
ax.set(ylabel='F1',ylim=(0,1));ax.legend();fig.tight_layout()
fig.savefig('qml_fixed_scores.png');plt.close(fig)

버전: {'qiskit': '2.2.3', 'qiskit-machine-learning': '0.9.0', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1'}
학습/시험 크기: (18, 7) (6, 7)
학습/시험 라벨 수: [9, 9] [3, 3]


VQC Z RealAmplitudes 학습 평가 수: 40 최초/마지막 기록 손실: 1.011689 0.657872
{'model': 'VQC', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [2, 1]]


SQNN Z RealAmplitudes 학습 평가 수: 40 최초/마지막 기록 손실: 1.011689 0.657872
{'model': 'SQNN', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [2, 1]]


EQNN Z RealAmplitudes 학습 평가 수: 40 최초/마지막 기록 손실: 1.005793 0.834497
{'model': 'EQNN', 'accuracy': 0.5, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [3, 0]]


VQC Z EfficientSU2 학습 평가 수: 40 최초/마지막 기록 손실: 1.00662 0.806422
{'model': 'VQC', 'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [0, 3]]


SQNN Z EfficientSU2 학습 평가 수: 40 최초/마지막 기록 손실: 1.00662 0.806422
{'model': 'SQNN', 'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [0, 3]]


... (26 outputs left out)

SQNN Pauli TwoLocal 학습 평가 수: 40 최초/마지막 기록 손실: 1.073708 0.922282
{'model': 'SQNN', 'accuracy': 0.6667, 'precision': 1.0, 'recall': 0.3333, 'f1': 0.5}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [2, 1]]


EQNN Pauli TwoLocal 학습 평가 수: 40 최초/마지막 기록 손실: 1.034808 0.985798
{'model': 'EQNN', 'accuracy': 0.5, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [3, 0]]


VQC Pauli PauliTwoDesign 학습 평가 수: 40 최초/마지막 기록 손실: 0.93958 0.860645
{'model': 'VQC', 'accuracy': 0.8333, 'precision': 1.0, 'recall': 0.6667, 'f1': 0.8}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [1, 2]]


SQNN Pauli PauliTwoDesign 학습 평가 수: 40 최초/마지막 기록 손실: 0.93958 0.860645
{'model': 'SQNN', 'accuracy': 0.8333, 'precision': 1.0, 'recall': 0.6667, 'f1': 0.8}
혼동행렬 [[TN,FP],[FN,TP]]: [[3, 0], [1, 2]]
EQNN Pauli PauliTwoDesign 학습 평가 수: 40 최초/마지막 기록 손실: 0.968388 0.91609
{'model': 'EQNN', 'accuracy': 0.8333, 'precision': 0.75, 'recall': 1.0, 'f1': 0.8571}
혼동행렬 [[TN,FP],[FN,TP]]: [[2, 1], [0, 3]]
모델별 F1 요약:
       min  max    mean
model                  
EQNN   0.0  1.0  0.5937
SQNN   0.5  1.0  0.7825
VQC    0.5  1.0  0.7825


36개 구성의 F1 범위는 VQC와 SQNN에서 0.5~1.0, EQNN에서 0~1.0입니다. 평균 F1은 VQC와 SQNN에서 약 0.7825, EQNN에서 약 0.5937입니다. VQC와 SQNN의 일치는 설정된 회로·출력 규칙·학습 조건의 일치와 연결됩니다. EQNN은 고전 출력층의 두 파라미터가 추가됩니다.

시험 데이터가 6건이므로 F1=1.0은 해당 여섯 건의 분류 결과입니다. 표본 확대, 데이터 변화와 반복 실험으로 성능을 추가 확인합니다. 이 실습에는 36회의 학습이 포함되며 계산시간은 실행 환경에 따라 달라집니다.